### Chargement du DataFrame

In [76]:
import pandas as pd

df = pd.read_csv("../data/processed/dataset_annotated_regex.csv")

labels = ["qualité produit", "service livraison", "service client"]

df_annotated = df[df[labels].sum(axis=1) > 0]

### Distribution des classes

In [77]:
class_distribution = df_annotated[labels].sum().to_frame(name="nb_commentaires")
class_distribution["pourcentage"] = (
    class_distribution["nb_commentaires"] / len(df_annotated) * 100
)

class_distribution

,nb_commentaires,pourcentage
qualité produit,5440,47.382632
service livraison,6971,60.717708
service client,6117,53.279331


### Affichage d'avis au hasard

In [78]:
samples = []

pd.set_option("display.max_colwidth", None)

df_annotated = df_annotated.drop(columns=["client"], errors="ignore")

for label in labels:
    sample_df = df_annotated[df_annotated[label] == 1].sample(
        n=3,
        random_state=None
    )
    
    sample_df = sample_df.copy()
    sample_df["label_cible"] = label  # pour savoir pourquoi il est sélectionné
    
    samples.append(sample_df)

result = pd.concat(samples)

display_cols = result.rename(columns={
    "qualité produit": "produit",
    "service livraison": "livraison",
    "service client": "client"
})

display_cols = display_cols[[
    "label_cible",
    "produit",
    "livraison",
    "client",
    "clean_comment"
]]

display(display_cols)

# for _, row in display_cols.iterrows():
#     print("─" * 100)
#     print(f"Label cible : {row['label_cible']}")
#     print(f"Produit    : {row['produit']}")
#     print(f"Livraison  : {row['livraison']}")
#     print(f"Client    : {row['client']}")
#     print("\nCommentaire :")
#     print(row["clean_comment"])

,label_cible,produit,livraison,client,clean_comment
3802,qualité produit,1,0,1,manque deux articles a la commande qui ne seront pas renvoyés ... très déçue mais marchandise de bonne qualité et services clients correct
2081,qualité produit,1,0,1,déçue par ma commande : 3 chemisiers taille 38 : 1 seul avec la bonne taille ; un trop grand un autre trop petit et la qualité pas top ! dommage
7617,qualité produit,1,0,0,très rapide et conforme
6185,service livraison,0,1,0,rapide pour la livraison produit tel que prévu
13467,service livraison,0,1,0,"cliente depuis plusieurs années , je n'ai jamais eu aucun souci avec ce site . que ce soit les produits , les livraisons ou les retours pour remboursements ; je considère donc ce site comme très sérieux ."
12450,service livraison,0,1,1,"et encore , 0 étoile aurait été mieux.commande d ’ un iphone reconditionné fin décembre , livraison qui démarre et que je peux suivre ( chez colis privé , aussi nul que veepee ) , et qui s ’ arrête le 29 décembre.ça sent le colis volé , je contacte veepee , et là on m ’ annonce un délai de 21 jours pour recherche ! ! ! au bout de quelques jours , je renvoie un message et j ’ ai l ’ annonce deux jours plus tard : colis non retrouvé , on vous rembourse , vous aurez la somme entre 3 et 8 jours ... ça c ’ était le 13 janvier.on est le 27 janvier , je n ’ ai toujours rien reçu.par contre , pour encaisser la somme , veepee n ’ a pas attendu la livraison ! ! ! je ne commanderai plus sur ce site"
2176,service client,0,1,1,"voleur catastrophique cette société surtout ne commandez jamais rien de cher car il ne respecte pas les délais de remboursements ! ! ! j'ai passé une commande de vaisselle guy degrenne d'une valeur de 271euros en recevant la vaisselle j'ai eu un choc , la vaisselle était uniquement constitué de rebuts.j ai donc tout retourné et depuis impossible d'être remboursé , il disent avoir reçu un premier colis , puis les autres soit disant apres `` bizarre '' il a fallut que je prouve que j'avais tout envoyé sinon c'était dans le baba ( gardez bien toutes les preuves d'envoies ) puis depuis ils changent sans cesse les délais de remboursement ... résultat je craque car depuis le 17 juin ou j'ai retourné toute la commande je n ai toujours pas ete remboursee ! ils mentent , racontent n importe quoi , je finis pas me dire que je vais devoir les poursuivre pour être remboursée ! ils travaillent tranquillent avoir l'argent des clients ! ! ! une honte"
12877,service client,0,1,1,a évitez ! j ’ ai retournée 3 jean lévis car sa ne correspondais pas le 7 mai . aujourd ’ hui nous somme le 7 juillet je suis toujours dans l ’ attente du remboursement . j ’ ai eu diffèrent personne au sav vepee qui me confirme la réception des colis . je leur demande alors le reboursement car d ’ après leur politique le remboursement se fait entre 3 et 8 jour après réception . ils me répondent qu ’ il me faut leur envoyer le bon de retour . a savoir que cela fais 2 mois donc bon de retour je ne l ’ ai plus . j ’ insiste que le faite que tous les conseiller que j ’ ai eu au téléphone un 0800 ligne payante en plus m ’ ont confirmer que ils avaient bien reçu les articles retour . inconherence et tout est mis en œuvre pour évitez de rembourser la clientèle . quand il réceptionne le colis et il ne vous informe pas . donc vous rester dans l ’ ignorance . quand vous vous rendez compte du délaie ( vous avez sûrement déjà perdu le bon retour relay pendant ce temps vous ete endormi et quand soudain vous revenez vers eux tout ce complique . en bref je ne commande plus chez eux .
3446,service client,0,0,1,toujours aussi satisfaite de mes achats et service après vente impeccable . à acheter sans hésiter


### Création des variables X_test/y

In [79]:
X_text = df["clean_comment"].values
y = df[[
    "qualité produit",
    "service livraison",
    "service client"
]].values

### Encoding de X_text

In [80]:
from sentence_transformers import SentenceTransformer

model_emb = SentenceTransformer(
    "dangvantuan/french-document-embedding",
    trust_remote_code=True
)

X_embeddings = model_emb.encode(
    X_text,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/472 [00:00<?, ?it/s]

### Séparation des données en train/test

In [81]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_embeddings,
    y,
    test_size=0.2,
    random_state=42
)

### Création du modèle

In [82]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

log_reg = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    max_iter=1000,
    class_weight="balanced"
)

clf = OneVsRestClassifier(log_reg)

### CrossVal

In [83]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, f1_score

scorer = make_scorer(f1_score, average="micro")

cv_scores = cross_val_score(
    clf,
    X_embeddings,
    y,
    cv=5,
    scoring=scorer,
    n_jobs=-1
)

print("F1 micro par fold :", cv_scores)
print("F1 micro moyen   :", cv_scores.mean())
print("Écart-type       :", cv_scores.std())

# la validation croisée met en évidence l'instabilité du modèle (ecart-type élevé : 0.078)

F1 micro par fold : [0.85271318 0.88425282 0.8725004  0.8750199  0.84805954]
F1 micro moyen   : 0.8665091685183292
Écart-type       : 0.013812187657302206


### Entrainement du modèle et calcul des probas

In [84]:
clf.fit(X_train, y_train)

,estimator,LogisticRegre...max_iter=1000)
,n_jobs,None
,verbose,0
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None


In [85]:
y_proba = clf.predict_proba(X_test)

### Affichage des résultats

In [86]:
from sklearn.metrics import classification_report

threshold = 0.5

y_pred = (y_proba >= threshold).astype(int)

print(classification_report(
    y_test,
    y_pred,
    target_names=[
        "qualité produit",
        "service livraison",
        "service client"
    ]
))

                   precision    recall  f1-score   support

  qualité produit       0.75      0.82      0.79      1086
service livraison       0.90      0.91      0.91      1374
   service client       0.86      0.93      0.89      1217

        micro avg       0.84      0.89      0.86      3677
        macro avg       0.84      0.89      0.86      3677
     weighted avg       0.84      0.89      0.87      3677
      samples avg       0.67      0.67      0.66      3677



c:\IA\trustpilot\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\IA\trustpilot\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\IA\trustpilot\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [87]:
from sklearn.metrics import f1_score

f1 = f1_score(y_test, y_pred, average=None)
for label, score in zip(
    ["qualité produit", "service livraison", "service client"], f1
):
    print(label, score)

qualité produit 0.7852437417654808
service livraison 0.90572878897752
service client 0.8921374950612406


In [91]:
import numpy as np

labels = ["qualité produit", "service livraison", "service client"]

threshold = 0.40

thresholds = {
    "qualité produit": threshold,
    "service livraison": threshold,
    "service client": threshold
}

y_pred = np.zeros_like(y_proba, dtype=int)

for i, label in enumerate(labels):
    y_pred[:, i] = (y_proba[:, i] >= thresholds[label]).astype(int)

from sklearn.metrics import confusion_matrix

for i, label in enumerate(labels):
    tn, fp, fn, tp = confusion_matrix(
        y_test[:, i],
        y_pred[:, i]
    ).ravel()
    
    df_cm = pd.DataFrame(
        [[tn, fp], [fn, tp]],
        index=["Vrai 0", "Vrai 1"],
        columns=["Prédit 0", "Prédit 1"]
    )
    
    print(f"\n{label}")
    display(df_cm)




qualité produit


,Prédit 0,Prédit 1
Vrai 0,1454,478
Vrai 1,126,960



service livraison


,Prédit 0,Prédit 1
Vrai 0,1452,192
Vrai 1,83,1291



service client


,Prédit 0,Prédit 1
Vrai 0,1555,246
Vrai 1,59,1158
